# Modeling 5 Computer Science Concepts with Probability, Markov Chains, Poisson Processes, and Renewal Theory

This notebook uses the main ideas developed across the book chapters:

- probability spaces, random variables, distributions, conditional probability
- expectation, variance, conditional expectation, independence
- Bernoulli processes and sums of independent random variables
- Poisson processes and exponential interarrival times
- discrete-time Markov chains and limiting distributions
- continuous-time Markov processes
- renewal, regenerative, and Markov-renewal thinking

We model five CS-related systems:

1. **Hash-table collisions and load distribution**
2. **Retry loops, exponential backoff, and time to success**
3. **Web/API server queues as an M/M/1 Markov process**
4. **Cache hit rates under a Markov request model**
5. **Failure, repair, checkpointing, and regenerative availability**

Each section follows the same discipline:

1. define the sample space and random variables,
2. state the modeling assumptions,
3. derive a closed-form result where possible,
4. simulate to check the formula,
5. explain what the model hides.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

def show_table(df, n=10):
    # Small helper to display a neat rounded table.
    display(df.head(n).round(4))

# 1. Hash-table collisions and load distribution

## CS concept

A hash table maps keys into `m` buckets. If `n` keys are hashed independently and uniformly, the bucket occupancies are random.

This is the classical “balls into bins” model:

- keys are balls,
- buckets are bins,
- a collision occurs when two or more keys land in the same bucket.

## Probability model

Let

\[
X_j = \text{number of keys in bucket } j,\qquad j=1,\ldots,m.
\]

For a fixed bucket \(j\),

\[
X_j \sim \operatorname{Binomial}(n, 1/m).
\]

So

\[
P(X_j=k) = \binom{n}{k}\left(\frac1m\right)^k\left(1-\frac1m\right)^{n-k}.
\]

The expected load per bucket is

\[
E[X_j] = \frac{n}{m}.
\]

For large \(m,n\), with \(\lambda = n/m\) fixed, the binomial distribution is well-approximated by

\[
X_j \approx \operatorname{Poisson}(\lambda),
\qquad
P(X_j=k) \approx e^{-\lambda}\frac{\lambda^k}{k!}.
\]

## Derivation: expected number of occupied buckets

Define an indicator random variable

\[
I_j =
\begin{cases}
1, & \text{bucket }j\text{ is nonempty},\\
0, & \text{otherwise}.
\end{cases}
\]

Then the total number of occupied buckets is

\[
O = \sum_{j=1}^m I_j.
\]

Now

\[
P(I_j=1) = 1 - P(X_j=0)
= 1-\left(1-\frac1m\right)^n.
\]

By linearity of expectation,

\[
E[O]
= \sum_{j=1}^m E[I_j]
= m\left[1-\left(1-\frac1m\right)^n\right].
\]

The expected number of colliding keys beyond the first key in each occupied bucket is

\[
E[\text{extra keys due to collisions}]
= n - E[O].
\]

In [ ]:
def simulate_hashing(n=10_000, m=8_000, trials=400):
    occupied = []
    max_load = []
    extra_collision_keys = []
    occupancy_samples = []

    for _ in range(trials):
        buckets = rng.integers(0, m, size=n)
        counts = np.bincount(buckets, minlength=m)
        occupied.append(np.sum(counts > 0))
        max_load.append(np.max(counts))
        extra_collision_keys.append(n - np.sum(counts > 0))
        occupancy_samples.extend(counts[:min(m, 200)])

    return {
        "occupied": np.array(occupied),
        "max_load": np.array(max_load),
        "extra_collision_keys": np.array(extra_collision_keys),
        "occupancy_samples": np.array(occupancy_samples),
    }

n, m = 10_000, 8_000
sim = simulate_hashing(n=n, m=m, trials=500)

lambda_load = n / m
expected_occupied = m * (1 - (1 - 1/m)**n)
expected_extra = n - expected_occupied

summary = pd.DataFrame({
    "quantity": [
        "load factor lambda=n/m",
        "E[occupied buckets] formula",
        "occupied buckets simulation mean",
        "E[extra collision keys] formula",
        "extra collision keys simulation mean",
        "max bucket load simulation mean",
    ],
    "value": [
        lambda_load,
        expected_occupied,
        sim["occupied"].mean(),
        expected_extra,
        sim["extra_collision_keys"].mean(),
        sim["max_load"].mean(),
    ]
})
show_table(summary, 10)

In [ ]:
samples = sim["occupancy_samples"]
max_k = int(np.percentile(samples, 99.7))
ks = np.arange(0, max_k + 1)

empirical = np.array([(samples == k).mean() for k in ks])
poisson = np.exp(-lambda_load) * np.array([lambda_load**k / math.factorial(k) for k in ks])

plt.figure(figsize=(8, 4.5))
plt.bar(ks - 0.2, empirical, width=0.4, label="simulation")
plt.bar(ks + 0.2, poisson, width=0.4, label=f"Poisson({lambda_load:.2f})")
plt.xlabel("keys in a bucket")
plt.ylabel("probability")
plt.title("Hash bucket occupancy: simulation vs Poisson approximation")
plt.legend()
plt.show()

## Operational interpretation

The Poisson approximation tells you what “random hashing noise” looks like. If real bucket occupancies are much worse than this, the problem is probably not ordinary randomness; it may be:

- a poor hash function,
- correlated keys,
- adversarial keys,
- modulo bias,
- poor resizing policy.

The model gives a baseline for what healthy randomness should look like.

# 2. Retry loops, exponential backoff, and time to success

## CS concept

Many distributed systems perform an operation repeatedly until it succeeds:

- RPC retry after timeout,
- database transaction retry after conflict,
- CAS loop retry after contention,
- packet retransmission.

A first model is a Bernoulli process.

At each attempt:

\[
X_i =
\begin{cases}
1, & \text{attempt }i\text{ succeeds},\\
0, & \text{attempt }i\text{ fails}.
\end{cases}
\]

Assume attempts are independent and

\[
P(X_i=1)=p,\qquad P(X_i=0)=q=1-p.
\]

Let

\[
T = \min\{i\ge 1: X_i=1\}
\]

be the attempt number of the first success. Then \(T\) has the geometric distribution:

\[
P(T=k)=q^{k-1}p,\qquad k=1,2,\ldots
\]

and

\[
E[T]=\frac1p,
\qquad
\operatorname{Var}(T)=\frac{q}{p^2}.
\]

## Proof of \(E[T]=1/p\)

Use the tail-sum formula for a nonnegative integer-valued random variable:

\[
E[T] = \sum_{n=0}^{\infty} P(T>n).
\]

The event \(T>n\) means the first \(n\) attempts all failed, so

\[
P(T>n)=q^n.
\]

Therefore

\[
E[T]
= \sum_{n=0}^{\infty} q^n
= \frac{1}{1-q}
= \frac1p.
\]

In [ ]:
def simulate_retries(p=0.3, max_attempts=10, trials=200_000):
    T = rng.geometric(p, size=trials)
    success_with_cap = T <= max_attempts
    attempts_used = np.minimum(T, max_attempts)
    return T, success_with_cap, attempts_used

p = 0.3
max_attempts = 8
T, success_with_cap, attempts_used = simulate_retries(p=p, max_attempts=max_attempts)

summary = pd.DataFrame({
    "quantity": [
        "p",
        "E[T] formula",
        "E[T] simulation",
        f"P(success within {max_attempts}) formula",
        f"P(success within {max_attempts}) simulation",
        f"E[attempts used with cap {max_attempts}] simulation",
    ],
    "value": [
        p,
        1/p,
        T.mean(),
        1 - (1-p)**max_attempts,
        success_with_cap.mean(),
        attempts_used.mean(),
    ],
})
show_table(summary, 10)

In [ ]:
ks = np.arange(1, 16)
pmf = (1-p)**(ks-1) * p
emp = np.array([(T == k).mean() for k in ks])

plt.figure(figsize=(8, 4.5))
plt.bar(ks - 0.2, emp, width=0.4, label="simulation")
plt.bar(ks + 0.2, pmf, width=0.4, label="geometric formula")
plt.xlabel("attempt number of first success")
plt.ylabel("probability")
plt.title("Retry loop: first-success distribution")
plt.legend()
plt.show()

## Adding exponential backoff

Suppose the retry waits are

\[
b_0,\; b_1,\; b_2,\ldots
\]

where

\[
b_i = \min(b_{\max}, b_{\min}2^i).
\]

If the \(k\)-th attempt succeeds, the elapsed waiting time before success is

\[
W(T) = \sum_{i=0}^{T-2} b_i.
\]

This is a function of the geometric stopping time \(T\). The expected waiting time is

\[
E[W(T)] = \sum_{k=1}^{\infty}
\left(\sum_{i=0}^{k-2} b_i\right)q^{k-1}p.
\]

With a maximum retry cap, the sum is finite.

In [ ]:
def backoff_schedule(max_attempts=8, base=0.050, cap=2.0):
    return np.array([min(cap, base * 2**i) for i in range(max_attempts - 1)])

def expected_backoff_wait(p=0.3, max_attempts=8, base=0.050, cap=2.0):
    q = 1 - p
    waits = backoff_schedule(max_attempts, base, cap)
    expected = 0.0
    rows = []
    for k in range(1, max_attempts + 1):
        prob_success_at_k = (q**(k-1)) * p
        wait_before_k = waits[:max(0, k-1)].sum()
        expected += prob_success_at_k * wait_before_k
        rows.append((k, prob_success_at_k, wait_before_k, prob_success_at_k * wait_before_k))
    prob_fail_all = q**max_attempts
    wait_if_fail = waits.sum()
    return expected, prob_fail_all, wait_if_fail, pd.DataFrame(
        rows, columns=["success_attempt", "probability", "wait_before_success_sec", "contribution_sec"]
    )

expected_wait, prob_fail_all, wait_if_fail, df_backoff = expected_backoff_wait(
    p=p, max_attempts=max_attempts, base=0.050, cap=2.0
)
show_table(df_backoff, 10)

print(f"Expected wait before success, counting only successful paths: {expected_wait:.4f} seconds")
print(f"Probability all attempts fail: {prob_fail_all:.4f}")
print(f"Wait paid before giving up if all fail: {wait_if_fail:.4f} seconds")

## Operational interpretation

Retries reduce visible failure probability:

\[
P(\text{all }r\text{ attempts fail}) = q^r.
\]

But retries also increase latency, load, and correlation. The independence assumption is often false: if the service is overloaded, many attempts fail together. In that case, retries can amplify overload unless backoff, jitter, budgets, and circuit breakers are used.

# 3. Web/API server queue as an M/M/1 Markov process

## CS concept

A server receives requests and processes them one at a time.

A standard model is the **M/M/1 queue**:

- arrivals form a Poisson process with rate \(\lambda\),
- service times are exponential with rate \(\mu\),
- one server,
- infinite queue,
- first-come-first-served.

Let

\[
X_t = \text{number of jobs in the system at time }t.
\]

Then \(X_t\) is a continuous-time Markov process on states

\[
0,1,2,\ldots
\]

with transitions

\[
i \to i+1 \quad \text{at rate } \lambda,
\]

and

\[
i \to i-1 \quad \text{at rate } \mu
\quad\text{for } i\ge 1.
\]

## Stationary distribution derivation

In equilibrium, the probability flow from \(i\) to \(i+1\) should match the probability flow from \(i+1\) to \(i\):

\[
\pi_i \lambda = \pi_{i+1}\mu.
\]

Thus

\[
\pi_{i+1} = \rho \pi_i,
\qquad
\rho = \frac{\lambda}{\mu}.
\]

So

\[
\pi_i = \rho^i \pi_0.
\]

Normalize:

\[
1 = \sum_{i=0}^{\infty}\pi_i
= \pi_0 \sum_{i=0}^{\infty}\rho^i
= \frac{\pi_0}{1-\rho},
\]

provided \(\rho<1\). Therefore

\[
\pi_0 = 1-\rho,
\qquad
\pi_i = (1-\rho)\rho^i.
\]

The expected number of jobs in the system is

\[
E[X] = \sum_{i=0}^{\infty} i(1-\rho)\rho^i
= \frac{\rho}{1-\rho}.
\]

This explosion as \(\rho\to 1\) is the core queueing lesson: utilization near 100% creates huge queues.

In [ ]:
def simulate_mm1(lam=8.0, mu=10.0, horizon=10_000.0, warmup=500.0):
    t = 0.0
    x = 0
    area_after_warmup = 0.0
    time_after_warmup = 0.0
    arrivals = 0
    departures = 0
    states_sample_t = []
    states_sample_x = []

    while t < horizon:
        rate = lam + (mu if x > 0 else 0.0)
        dt = rng.exponential(1 / rate)
        next_t = t + dt

        start = max(t, warmup)
        end = min(next_t, horizon)
        if end > start:
            area_after_warmup += x * (end - start)
            time_after_warmup += end - start

        t = next_t
        if t > horizon:
            break

        if rng.random() < lam / rate:
            x += 1
            arrivals += 1
        else:
            x -= 1
            departures += 1

        if len(states_sample_t) < 5000 and t > warmup:
            states_sample_t.append(t)
            states_sample_x.append(x)

    return {
        "mean_number": area_after_warmup / time_after_warmup,
        "arrivals": arrivals,
        "departures": departures,
        "sample_t": np.array(states_sample_t),
        "sample_x": np.array(states_sample_x),
    }

lam, mu = 8.0, 10.0
rho = lam / mu
mm1 = simulate_mm1(lam=lam, mu=mu)

summary = pd.DataFrame({
    "quantity": [
        "lambda arrival rate",
        "mu service rate",
        "rho=lambda/mu",
        "E[number in system] formula",
        "E[number in system] simulation",
        "P(system empty) formula",
    ],
    "value": [
        lam,
        mu,
        rho,
        rho/(1-rho),
        mm1["mean_number"],
        1-rho,
    ]
})
show_table(summary, 10)

In [ ]:
plt.figure(figsize=(9, 4))
plt.step(mm1["sample_t"][:400] - mm1["sample_t"][0], mm1["sample_x"][:400], where="post")
plt.xlabel("time after sample start")
plt.ylabel("number of jobs")
plt.title("M/M/1 queue sample path")
plt.show()

In [ ]:
max_state = 20
states_num = np.arange(max_state + 1)
pi_queue = (1-rho) * rho**states_num

plt.figure(figsize=(8, 4.5))
plt.bar(states_num, pi_queue)
plt.xlabel("number of jobs in system")
plt.ylabel("stationary probability")
plt.title("M/M/1 stationary distribution")
plt.show()

## Operational interpretation

The model predicts:

\[
E[X] = \frac{\rho}{1-\rho}.
\]

So moving utilization from 80% to 90% does not increase queue length by 12.5%; it more than doubles it:

\[
\frac{0.9/(1-0.9)}{0.8/(1-0.8)} = \frac{9}{4}.
\]

This is why servers need headroom. Tail latency gets bad well before average CPU reaches 100%.

# 4. Cache hit rates under a Markov request model

## CS concept

A cache stores recently or frequently used objects. The hit rate depends heavily on request locality.

Independent request models are often too crude. A better minimal model is a Markov chain over requested objects:

\[
R_n \in \{A,B,C\}.
\]

The next request depends on the current request:

\[
P(R_{n+1}=j \mid R_n=i) = P(i,j).
\]

We model a cache of size 1 that always stores the most recently requested object. Then request \(n+1\) is a hit exactly when

\[
R_{n+1}=R_n.
\]

So the long-run hit rate is

\[
\sum_i \pi(i)P(i,i),
\]

where \(\pi\) is the stationary distribution of the request Markov chain.

## Stationary distribution

A stationary distribution \(\pi\) satisfies

\[
\pi = \pi P,
\qquad
\sum_i \pi_i = 1.
\]

Once the request chain reaches stationarity, the long-run fraction of requests in state \(i\) is \(\pi_i\). Therefore, the long-run size-1 cache hit rate is

\[
H_1 = \sum_i \pi_i P(i,i).
\]

In [ ]:
request_states = ["A", "B", "C"]

P_cache = np.array([
    [0.80, 0.15, 0.05],
    [0.20, 0.70, 0.10],
    [0.30, 0.20, 0.50],
])

def stationary_distribution(P):
    n = P.shape[0]
    A = (P.T - np.eye(n))
    A[-1, :] = 1.0
    b = np.zeros(n)
    b[-1] = 1.0
    return np.linalg.solve(A, b)

pi_cache = stationary_distribution(P_cache)
hit_rate_size1 = np.sum(pi_cache * np.diag(P_cache))

pd.DataFrame({
    "state": request_states,
    "stationary_probability": pi_cache,
    "self_transition_probability": np.diag(P_cache),
    "hit_rate_contribution": pi_cache * np.diag(P_cache),
}).round(4)

In [ ]:
def simulate_markov_requests(P, n=200_000, start=0):
    x = np.empty(n, dtype=int)
    x[0] = start
    for k in range(1, n):
        x[k] = rng.choice(P.shape[0], p=P[x[k-1]])
    return x

requests = simulate_markov_requests(P_cache)
hits_size1 = requests[1:] == requests[:-1]
sim_hit_rate_size1 = hits_size1.mean()

summary = pd.DataFrame({
    "quantity": [
        "stationary hit rate formula for size-1 MRU cache",
        "simulation hit rate for size-1 MRU cache",
    ],
    "value": [hit_rate_size1, sim_hit_rate_size1]
})
show_table(summary, 10)

## Size-2 LRU cache as a Markov chain on cache states

For a cache of size 2 over objects \(A,B,C\), the cache state must include order.

Let state `(x, y)` mean:

- `x` is most recently used,
- `y` is second most recently used.

A request for:

- `x`: hit, state stays `(x, y)`,
- `y`: hit, state becomes `(y, x)`,
- some new object `z`: miss, state becomes `(z, x)`.

The pair `(current request process, cache contents)` is Markov. We simulate it directly.

In [ ]:
def simulate_lru_cache_markov_requests(P, cache_size=2, n=200_000, start=0):
    reqs = simulate_markov_requests(P, n=n, start=start)
    cache = []
    hits = []

    for r in reqs:
        hit = r in cache
        hits.append(hit)
        if hit:
            cache.remove(r)
            cache.insert(0, r)
        else:
            cache.insert(0, r)
            cache = cache[:cache_size]

    return reqs, np.array(hits)

_, hits_size2 = simulate_lru_cache_markov_requests(P_cache, cache_size=2)
summary = pd.DataFrame({
    "cache": ["size 1 MRU", "size 2 LRU"],
    "hit_rate": [sim_hit_rate_size1, hits_size2[1:].mean()]
})
show_table(summary)

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(summary["cache"], summary["hit_rate"])
plt.ylim(0, 1)
plt.ylabel("hit rate")
plt.title("Cache hit rate under Markov request locality")
plt.show()

## Operational interpretation

The important variable is not only popularity. It is **temporal dependence**.

Two workloads can have the same marginal frequencies for objects but very different cache hit rates:

- independent requests: weak locality,
- Markov requests with large \(P(i,i)\): strong locality,
- cyclic requests: may defeat small caches.

A Markov chain exposes locality through transition probabilities, not just frequencies.

# 5. Failure, repair, checkpointing, and regenerative availability

## CS concept

Consider a service or worker that alternates between:

- **up** periods,
- **down/repair** periods.

This is a renewal or regenerative process. Each cycle restarts probabilistically after repair.

Let

\[
U_1, U_2,\ldots
\]

be i.i.d. up times, and

\[
D_1, D_2,\ldots
\]

be i.i.d. down times. One cycle length is

\[
C_i = U_i + D_i.
\]

The long-run availability is the fraction of time spent up:

\[
A = \frac{E[U]}{E[U]+E[D]}.
\]

This is a renewal-reward result: reward rate equals expected reward per cycle divided by expected cycle length.

## Proof sketch: renewal reward idea

For \(n\) completed cycles, total up time is

\[
U_1+\cdots+U_n,
\]

and total elapsed cycle time is

\[
(U_1+D_1)+\cdots+(U_n+D_n).
\]

The observed availability after many cycles is approximately

\[
\frac{U_1+\cdots+U_n}
{(U_1+D_1)+\cdots+(D_n+U_n)}.
\]

By the law of large numbers,

\[
\frac{U_1+\cdots+U_n}{n}\to E[U],
\qquad
\frac{(U_1+D_1)+\cdots+(U_n+D_n)}{n}\to E[U]+E[D].
\]

Therefore,

\[
A = \frac{E[U]}{E[U]+E[D]}.
\]

In [ ]:
def simulate_availability(mean_up=100.0, mean_down=5.0, horizon=200_000.0):
    t = 0.0
    up_time = 0.0
    cycles = 0
    times = []
    availability_so_far = []

    while t < horizon:
        u = rng.exponential(mean_up)
        d = rng.exponential(mean_down)

        up_end = min(t + u, horizon)
        up_time += max(0.0, up_end - t)
        t = min(t + u + d, horizon)
        cycles += 1

        if cycles % 50 == 0 and t > 0:
            times.append(t)
            availability_so_far.append(up_time / t)

    return up_time / horizon, np.array(times), np.array(availability_so_far)

mean_up, mean_down = 100.0, 5.0
availability_formula = mean_up / (mean_up + mean_down)
availability_sim, ts, avs = simulate_availability(mean_up, mean_down)

summary = pd.DataFrame({
    "quantity": [
        "mean up time",
        "mean down time",
        "availability formula",
        "availability simulation",
    ],
    "value": [mean_up, mean_down, availability_formula, availability_sim]
})
show_table(summary, 10)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(ts, avs)
plt.axhline(availability_formula, linestyle="--", label="formula")
plt.xlabel("time")
plt.ylabel("observed availability")
plt.title("Regenerative availability converges to renewal-reward ratio")
plt.legend()
plt.show()

## Checkpointing model

Now consider a long-running job subject to failures.

Assume:

- failures arrive as a Poisson process with rate \(\lambda\),
- checkpoint cost is \(c\) seconds,
- checkpoint interval is \(\tau\) seconds,
- after a failure, work since the last checkpoint is lost.

A simple approximation for overhead per unit useful work is:

\[
\text{overhead}(\tau)
\approx
\frac{c}{\tau} + \frac{\lambda \tau}{2}.
\]

Why?

- checkpointing costs \(c\) every \(\tau\) units, so checkpoint overhead rate is \(c/\tau\);
- if a failure occurs, the expected lost work since the last checkpoint is about \(\tau/2\);
- failures occur at rate \(\lambda\), so lost-work overhead rate is \(\lambda\tau/2\).

Minimize

\[
h(\tau)=\frac{c}{\tau}+\frac{\lambda\tau}{2}.
\]

Set derivative to zero:

\[
h'(\tau)=-\frac{c}{\tau^2}+\frac{\lambda}{2}=0.
\]

Thus

\[
\tau^* = \sqrt{\frac{2c}{\lambda}}.
\]

This is a practical renewal-reward style tradeoff: checkpoint too often and overhead dominates; checkpoint too rarely and failure loss dominates.

In [ ]:
def checkpoint_overhead(tau, c, lam):
    return c / tau + lam * tau / 2

c = 30.0
mean_time_to_failure = 6 * 3600
lam_fail = 1 / mean_time_to_failure

tau_star = math.sqrt(2 * c / lam_fail)

taus = np.linspace(60, 4 * 3600, 400)
overheads = checkpoint_overhead(taus, c, lam_fail)

summary = pd.DataFrame({
    "quantity": [
        "checkpoint cost c seconds",
        "mean time to failure seconds",
        "failure rate lambda",
        "optimal checkpoint interval tau* seconds",
        "optimal checkpoint interval minutes",
        "minimum overhead fraction approximation",
    ],
    "value": [
        c,
        mean_time_to_failure,
        lam_fail,
        tau_star,
        tau_star/60,
        checkpoint_overhead(tau_star, c, lam_fail),
    ]
})
show_table(summary, 10)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(taus / 60, overheads)
plt.axvline(tau_star / 60, linestyle="--", label=f"tau* = {tau_star/60:.1f} min")
plt.xlabel("checkpoint interval tau, minutes")
plt.ylabel("overhead per unit useful work")
plt.title("Checkpoint interval tradeoff")
plt.legend()
plt.show()

## Operational interpretation

This model captures a common systems pattern:

- very frequent checkpoints waste time,
- very rare checkpoints lose too much work on failure,
- the optimum depends on checkpoint cost and failure rate.

The formula is not exact for every system. It assumes memoryless failures, fixed checkpoint cost, and average loss \(\tau/2\). But it is a strong first-order model and usually gives the right scaling:

\[
\tau^* \propto \sqrt{c}
\qquad\text{and}\qquad
\tau^* \propto \frac{1}{\sqrt{\lambda}}.
\]

So if checkpoints become 4x more expensive, checkpoint half as often. If failures become 4x more frequent, checkpoint twice as often.

# Summary: mapping CS concepts to stochastic-process tools

| CS concept | Random model | Main result | Practical lesson |
|---|---:|---:|---|
| Hash-table collisions | Binomial / Poisson approximation | \(X_j\sim\mathrm{Binomial}(n,1/m)\approx\mathrm{Poisson}(n/m)\) | Randomness gives a baseline for healthy load distribution |
| Retry loops | Bernoulli / geometric | \(E[T]=1/p\), \(P(\text{fail all }r)=q^r\) | Retries trade failure probability for latency and load |
| Server queues | Poisson arrivals + CTMC | M/M/1: \(\pi_i=(1-\rho)\rho^i\), \(E[X]=\rho/(1-\rho)\) | Queue length explodes near saturation |
| Cache locality | Markov chain | hit rate \(=\sum_i \pi_iP(i,i)\) for size-1 MRU | Temporal dependence matters more than raw popularity |
| Failure/repair/checkpointing | Renewal reward | availability \(=E[U]/(E[U]+E[D])\), \(\tau^*=\sqrt{2c/\lambda}\) | Long-run ratios are cycle rewards over cycle lengths |

## How to use this notebook

For each section, try changing the parameters:

- hash table: `n`, `m`
- retry loop: `p`, `max_attempts`, backoff parameters
- queue: `lambda`, `mu`
- cache: transition matrix `P_cache`
- failure/checkpointing: `mean_up`, `mean_down`, `c`, `lambda`

Then rerun the cells and compare the formulas with simulations.